<a href="https://colab.research.google.com/github/claudyanepmarinho/Streaming-de-filmes/blob/main/Desafio1_API_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 Instalar

In [1]:
# Instala as bibliotecas usadas no guia.
!pip install -q gspread faker

Importações e Autenticação (service account)

In [2]:
# Importa as ferramentas para autenticação, geração de dados e sorteios.
import json, gspread
from google.colab import userdata
from faker import Faker
import random

# Lê a credencial salva nos Secrets e conecta à planilha.
info = json.loads(userdata.get("GCP_SA_JSON"))
gc = gspread.service_account_from_dict(info)
# Exibe o e-mail da conta autenticada.
print("Conectado como:", info["client_email"])

Conectado como: desafio1-python@streaming-de-filmes-513817.iam.gserviceaccount.com


 Abrir a planilha

In [3]:
# Informe o ID da planilha e o nome que será dado à aba.
SHEET_ID = "1vh82vpRPv8X6Rmsck8MpjJWWwYvk70SC2r_mctiMjjU"
NOME_TABELA = "filmes"   # nome da tabela na ficha do tema, ex.: "vagas"

# Abre a planilha e seleciona sua primeira aba.
sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1
# Renomeia a aba e exibe os dados da planilha.
ws.update_title(NOME_TABELA)     # a aba passa a ter o nome da tabela
print(sh.title, "|", ws.title, "|", sh.url)

Desafio_01_Streaming_Filmes | filmes | https://docs.google.com/spreadsheets/d/1vh82vpRPv8X6Rmsck8MpjJWWwYvk70SC2r_mctiMjjU


Cabeçalho – Tabela Dimensão

In [4]:
# Substitua pelos nomes EXATOS de coluna da ficha do seu tema
# Define as colunas que formarão o cabeçalho da tabela.
COLUNAS = ["filme_id", "titulo", "genero", "ano_lancamento", "duracao_min", "classificacao_indicativa", "diretor_ficticio"]

# Escreve o cabeçalho na primeira célula e confere o resultado.
ws.update(range_name="A1", values=[COLUNAS])
print("Cabeçalho confere:", ws.row_values(1) == COLUNAS)

Cabeçalho confere: True


 10 linhas de teste – Tabela Dimensão


In [5]:
# Configura os geradores aleatórios para produzir resultados reproduzíveis.
fake = Faker("pt_BR")
Faker.seed(42)
random.seed(42)

def gerar_linha(i):
    # Cria um registro fictício para as colunas definidas no esquema.
    # Personalize de acordo com o seu esquema de dados
    return {
        "filme_id": i,
        "titulo":random.choice(["Marcha para o Passado", "Quintas de Sorvete", "O Mistério da Janela Azul", "Domingo em Marte", "As Cartas que Nunca Chegaram", "Café com Tempestade", "A Última Goiabada", "Quando as Nuvens Dançam", "O Vizinho do Fim do Mundo", "Terror em Capelinha"]),
        "genero": random.choice(["ação", "comédia", "drama", "terror", "documentário", "animação", "ficção científica", "romance"]),
        "ano_lancamento": random.randint(1985, 2024),
        "duracao_min": random.randint(75, 184),
        "classificacao_indicativa": random.choice(["L", "10", "12", "14", "16", "18"]),
        "diretor_ficticio": fake.name()
    }

# Gera 10 registros numerados de 1 a 10.
linhas = [gerar_linha(i) for i in range(1, 11)]

# Organiza os valores na ordem do cabeçalho; nome de coluna incorreto gera KeyError.
valores = [[linha[c] for c in COLUNAS] for linha in linhas]
# Acrescenta os registros à planilha e informa quantos foram enviados.
ws.append_rows(valores, value_input_option="RAW")
print(len(valores), "linhas enviadas")

10 linhas enviadas


 Funções de envio de dados - Planilha

In [6]:
# Importa recursos para controlar pausas, acessar a planilha e tratar tabelas.
import time
import gspread
import pandas as pd

def df_para_valores(df):
    """DataFrame -> lista de listas com tipos nativos do Python."""
    # Trabalha em uma cópia para não alterar o DataFrame original.
    df = df.copy()
    for c in df.columns:
        # Converte datas para texto em um formato padronizado.
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.strftime("%Y-%m-%d %H:%M:%S")
        # Converte outros valores de data e hora para texto.
        elif df[c].dtype == object:
            df[c] = df[c].map(
                lambda v: v.isoformat() if hasattr(v, "isoformat") else v)
    # Substitui valores ausentes por texto vazio e retorna linhas de valores.
    return df.astype(object).where(df.notna(), "").values.tolist()

def enviar(sh, df, nome_aba, lote=5000, pausa=2):
    """Escreve cabeçalho e linhas numa aba, em lotes, e confere a contagem."""
    # Formata números com ponto como separador de milhar.
    n = lambda x: f"{x:,}".replace(",", ".")
    try:
        # Reutiliza a aba existente e apaga seu conteúdo anterior.
        ws = sh.worksheet(nome_aba)
        ws.clear()
    except gspread.WorksheetNotFound:
        # Cria uma aba caso ela ainda não exista.
        ws = sh.add_worksheet(title=nome_aba, rows=len(df) + 1,
                              cols=len(df.columns))
    # Ajusta o tamanho da aba e grava os nomes das colunas.
    ws.resize(rows=len(df) + 1, cols=len(df.columns))
    ws.update(range_name="A1", values=[list(df.columns)])
    # Converte o DataFrame e envia as linhas em lotes.
    valores = df_para_valores(df)
    for i in range(0, len(valores), lote):
        ws.append_rows(valores[i:i + lote], value_input_option="RAW")
        print(f"  {nome_aba}: {n(min(i + lote, len(valores)))} / {n(len(valores))}")
        time.sleep(pausa)               # respeita o limite de escritas/minuto
    # Confere se a quantidade de linhas gravadas corresponde à esperada.
    lidas = len(ws.col_values(1)) - 1   # lê de volta e confere
    if lidas != len(df):
        raise RuntimeError(f"Aba {nome_aba}: enviei {len(df)}, li {lidas}.")
    print(f"OK {nome_aba}: {n(len(df))} linhas na planilha")

def remover_aba_padrao(sh):
    """Apaga a aba vazia que o Google cria junto com a planilha nova."""
    # Localiza e remove a aba padrão somente se houver outra aba disponível.
    abas = sh.worksheets()
    padrao = ("Página1", "Sheet1", "Planilha1")
    for w in abas:
        if w.title in padrao and len(abas) > 1 and not w.get_all_values():
            sh.del_worksheet(w)

Carregar Bases de dados - Prontas

In [8]:
# Abre a opção de selecionar arquivos CSV no Colab e envia os arquivos.
from google.colab import files
files.upload()   # selecione os arquivos CSV do seu tema

# Carrega cada CSV em um DataFrame e associa-o ao nome da tabela.
# Altere o nome da tabela de acordo com a sua Base de dados
BASES = {
    "filmes": pd.read_csv("filmes.csv"),
    "visualizacoes": pd.read_csv("visualizacoes.csv"),
}
# Calcula e exibe a quantidade de linhas carregadas.
total = sum(len(d) for d in BASES.values())
print({n: len(d) for n, d in BASES.items()}, "| total:", total)

Saving visualizacoes.csv to visualizacoes.csv
Saving filmes.csv to filmes (1).csv
{'filmes': 10300, 'visualizacoes': 55000} | total: 65300


Opção B — Gerar as tabelas com Faker e numpy



In [ ]:
# # Importa ferramentas para criar tabelas e gerar dados fictícios.
# import numpy as np
# from faker import Faker

# # Define geradores aleatórios reproduzíveis.
# rng = np.random.default_rng(42)
# fake = Faker("pt_BR")
# Faker.seed(42)

# # ---- Tabela 1 (de referência) ----
# # Define a quantidade de registros da tabela de referência.
# N1 = 10_000
# # Cria IDs sequenciais e escolhe valores fictícios para as demais colunas.
# tabela_1 = pd.DataFrame({
#     "coluna_1": np.arange(1, N1 + 1),
#     "coluna_2": rng.choice(["A", "B", "C"], size=N1, p=[0.5, 0.3, 0.2]),
#     # ... uma entrada para CADA coluna da ficha ...
# })

# # ---- Tabela 2 (usa os IDs da tabela 1) ----
# # Define a quantidade de registros da tabela que referencia a primeira.
# N2 = 20_000
# # Cria IDs e escolhe, para cada linha, um ID existente na tabela 1.
# tabela_2 = pd.DataFrame({
#     "coluna_1": np.arange(1, N2 + 1),
#     "coluna_2": rng.integers(1, N1 + 1, size=N2),   # FK: ID da tabela 1
#     # ...
# })

# # Guarda as tabelas e a ordem esperada de suas colunas.
# BASES = {"nome_tabela_1": tabela_1, "nome_tabela_2": tabela_2}
# COLUNAS_FICHA = {
#     "nome_tabela_1": ["coluna_1", "coluna_2"],       # copie da ficha, na mesma ordem
#     "nome_tabela_2": ["coluna_1", "coluna_2"],
# }
# # Confere se as colunas de cada tabela seguem o esquema da ficha.
# for nome, df in BASES.items():
#     assert list(df.columns) == COLUNAS_FICHA[nome], f"Colunas erradas em {nome}"
# # Calcula e exibe a quantidade total de registros gerados.
# total = sum(len(d) for d in BASES.values())
# print({n: len(d) for n, d in BASES.items()}, "| total:", total)

ENVIAR

In [9]:
# Envia cada tabela para uma aba e remove a aba padrão, se estiver vazia.
for nome, df in BASES.items():
    enviar(sh, df, nome)        # uma aba por tabela
remover_aba_padrao(sh)

  filmes: 5.000 / 10.300
  filmes: 10.000 / 10.300
  filmes: 10.300 / 10.300
OK filmes: 10.300 linhas na planilha
  visualizacoes: 5.000 / 55.000
  visualizacoes: 10.000 / 55.000
  visualizacoes: 15.000 / 55.000
  visualizacoes: 20.000 / 55.000
  visualizacoes: 25.000 / 55.000
  visualizacoes: 30.000 / 55.000
  visualizacoes: 35.000 / 55.000
  visualizacoes: 40.000 / 55.000
  visualizacoes: 45.000 / 55.000
  visualizacoes: 50.000 / 55.000
  visualizacoes: 55.000 / 55.000
OK visualizacoes: 55.000 linhas na planilha


Validar a base na planilha

##Validar

In [10]:
# Soma e exibe a quantidade total de linhas das tabelas.
total = sum(len(d) for d in BASES.values())
print("Total de linhas:", f"{total:,}".replace(",", "."))
# Interrompe a execução se o volume mínimo não for atingido.
assert total >= 20_000, "Abaixo do mínimo (20.000 linhas somadas)."

# Confere os cabeçalhos e a quantidade de registros em cada aba.
for nome, df in BASES.items():
    w = sh.worksheet(nome)
    assert w.row_values(1) == list(df.columns), f"Cabeçalho diferente em {nome}"
    assert len(w.col_values(1)) - 1 == len(df), f"Contagem diferente em {nome}"
# Confirma que todas as verificações passaram.
print("A planilha confere com a base gerada.")

Total de linhas: 65.300
A planilha confere com a base gerada.
